#### CELDA DE CARGA DE IMAGENES (.npy)

In [2]:
import os
import glob
import numpy as np
from datetime import datetime

ruta_carpeta = '' # si lo hicieramos local poner la ubicación ...
busqueda = os.path.join(ruta_carpeta, '*.npy')
imagenes_npy = sorted(glob.glob(busqueda))

matrices_clorofila = []
fechas = []

for ruta_archivo in imagenes_npy:
    matriz = np.load(ruta_archivo)
    matrices_clorofila.append(matriz)

    nombre_archivo = os.path.basename(ruta_archivo)
    
    string_fecha = nombre_archivo.split('_')[1] 
    
    fecha = datetime.strptime(string_fecha, '%Y-%m-%d').date()
    fechas.append(fecha)

print(f"Cargadas {len(matrices_clorofila)} matrices clorofila")
print(f"Fechas: {fechas}")

Cargadas 0 matrices clorofila
Fechas: []


### Preprocessing

In [3]:
def preprocessing(matrices_clorofila, fechas):
  N = len(matrices_clorofila)
  H, W = matrices_clorofila[0].shape # caracteristicas propias de nuestras imágenes que
                            # definen la entrada al modelo
  mascara_agua = ~np.isnan(matrices_clorofila[0])

  matrices_limpias = [np.nan_to_num(m, nan=0.0) for m in matrices_clorofila] # convertimos todo nan a 0, mejor -1 o -9999?

  '''
  # Normalizo ahora todas las matrices, con min max (podemos explorar otras)
  max_clorofila = np.max(matrices_limpias)
  matrices_normalizadas = [(m - np.min(m)) / (max_clorofila - np.min(m)) for m in matrices_limpias]
  '''
  #|
  #-> esta es la primera normalización que hice, el tema es que habría que tomar el percentil 98 y 2 para el max y minimo
  #   de lo contrario los outliers (que los hay por errores de satelite) nos van a planchar todos los valores para abajo.

  pixeles_todos = np.concatenate([m[mascara_agua] for m in matrices_limpias])
  p_max = np.percentile(pixeles_todos, 98)
  p_min = np.percentile(pixeles_todos, 2)

  matrices_normalizadas = []

  for m in matrices_limpias:
        # min max
        m_norm = (m - p_min) / (p_max - p_min)

        # a todo aquellos valores fuera de los percentiles los tiramos a 0 o a 1
        m_norm = np.clip(m_norm, 0.0, 1.0)

        matrices_normalizadas.append(m_norm)



  deltas_t = [0] #primer delta t consideramos 0
  for i in range(1, N):
        delta_t_aux = (fechas[i] - fechas[i-1]).days
        deltas_t.append(delta_t_aux)

  # hay que normalizar también los delta_t, me parece lógica una normalización con el valor máximo

  max_delta_t = np.max(deltas_t)
  deltas_t_normalizados = [delta_t / max_delta_t for delta_t in deltas_t]

  dataset = np.zeros((N, H, W, 2))
  for i in range(N):
    dataset[i, :, :, 0] = matrices_normalizadas[i] * mascara_agua

    matriz_deltas_t = np.full((H,W), deltas_t_normalizados[i])
    dataset[i, :, :, 1] = matriz_deltas_t * mascara_agua


  return dataset, p_min, p_max, max_delta_t, mascara_agua

dataset_tensor, p_min, p_max, max_dt, mascara = preprocessing(matrices_clorofila, fechas)
# esto es un desempaquetado, para que queden las variables calculadas antes guardadas en memoria
print(f"Shape del dataset: {dataset_tensor.shape}")

NameError: name 'matrices_clorofila' is not defined

### Secuencias de entrenamiento

In [4]:
def crear_secuencias(dataset, sec_tamaño=14):
  X, Y = [],[]
  N = len(dataset)

  for i in range(N - sec_tamaño):
    X.append(dataset[i : i + sec_tamaño])
    Y.append(dataset[i+sec_tamaño, :, :, 0:1]) # -> el día 15 para test, ponemos 0:1 para que nos devuelva solo el canal de clorofila pero no perdamos las dimensiones

  return np.array(X), np.array(Y)

X_train, Y_train = crear_secuencias(dataset_tensor, sec_tamaño=14)

### Arquitectura y Entrenamiento Modelo

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def crear_modelo(sec_tamaño, H, W):
  modelo = models.Sequential()
  modelo.add(layers.Input(shape=(sec_tamaño, H, W, 2)))

  # primeras 4 capas ConvLSTM2D:

  # Capa 1
  modelo.add(layers.ConvLSTM2D(filters=32, kernel_size=(3,3), padding='same', return_sequences=True))
  modelo.add(layers.BatchNormalization())
  # Capa 2
  modelo.add(layers.ConvLSTM2D(filters=32, kernel_size=(3,3), padding='same', return_sequences=True))
  modelo.add(layers.BatchNormalization())
  # Capa 3
  modelo.add(layers.ConvLSTM2D(filters=32, kernel_size=(3,3), padding='same', return_sequences=True))
  modelo.add(layers.BatchNormalization())
  # Capa 4
  modelo.add(layers.ConvLSTM2D(filters=32, kernel_size=(3,3), padding='same', return_sequences=True))
  modelo.add(layers.BatchNormalization())
  # Capa 5
  modelo.add(layers.ConvLSTM2D(filters=32, kernel_size=(3,3), padding='same', return_sequences=True))
  modelo.add(layers.BatchNormalization())

  # Ultima capa:

  modelo.add(layers.Conv3D(filters=1, kernel_size=(3,3,3), activation='sigmoid', padding='same'))

  modelo.add(layers.Lambda(lambda x: x[:, -1, :, :, :])) #-> extraemos del tensor la capa con t = 15, o -1, que es lo mismo en python


  modelo.compile(optimizer='adam', loss='mse')

  return modelo

seq_len, H, W = X_train.shape[1], X_train.shape[2], X_train.shape[3]
modelo = crear_modelo(seq_len, H, W)
modelo.summary()


#### entrenamiento

In [ ]:
modelo.fit(X_train, Y_train, batch_size=4, epochs=50, validation_split=0.1)

### Predicción

In [5]:
def predecir(modelo, secuencia_inicial, mascara_agua, dias_a_predecir, max_dt, p_min, p_max):
  H, W = mascara_agua.shape
  predicciones = []

  secuencia_aux = secuencia_inicial.copy()

  delta_t_interno = 1 # el delta t entre predicciones (difiere de dias_a_predecir)
  delta_t_interno_norm = delta_t_interno / max_dt

  for dia in range(dias_a_predecir):
    print(f"El modelo está prediciendo el día {dia + 1} de {dias_a_predecir}")
    prediccion = modelo.predict(secuencia_aux, verbose=0)[0]

    #queremos ahora limpiar todo lo que el modelo pueda haber predicho que no haya sido agua
    prediccion = prediccion * mascara_agua[..., np.newaxis]

    # desnormalizamos:
    prediccion_real = (prediccion * (p_max - p_min)) + p_min
    prediccion_real = prediccion_real * mascara_agua[..., np.newaxis] #volvemos a hacer 0 pixeles indeseados
    predicciones.append(prediccion_real)

    nuevo_frame = np.zeros((1, 1, H, W, 2))

    # Canal 0: Clorofila predicha (normalizada)
    nuevo_frame[0, 0, :, :, 0] = prediccion[:, :, 0]
    # Canal 1: Delta T (normalizado)
    matriz_deltas_t = np.full((H, W), delta_t_interno_norm)
    nuevo_frame[0, 0, :, :, 1] = matriz_deltas_t * mascara_agua

    # dropeamos el dia mas viejo y agregamos el nuevo
    secuencia_aux = np.concatenate([secuencia_aux[:, 1:, :, :, :], nuevo_frame], axis=1)

  return np.array(predicciones)


In [ ]:
ultimos_14_dias = dataset_tensor[-14:]

secuencia_para_predecir = np.expand_dims(ultimos_14_dias, axis=0) # agregamos una dimension vacia para poder insertarla a predecir

dias_futuros = 5
resultado = predecir(
    modelo=modelo,
    secuencia_inicial=secuencia_para_predecir,
    mascara_agua=mascara,
    dias_a_predecir=dias_futuros,
    max_dt=max_dt,
    p_min=p_min,
    p_max=p_max
)

# va a generar una imágen por dia futuro predicho

for i, mapa in enumerate(resultado):
    # quitamos el canal extra para que vuelva a ser (H, W)
    imagen_resultante = mapa[:, :, 0]
    np.save(f"prediccion_clorofila_dia_{i+1}.npy", imagen_resultante)

### GRAFICO

In [ ]:
import matplotlib.pyplot as plt

ruta_npy = ".npy"  # ruta del npy a graficar
fecha_img = ""             # Título para el gráfico
fecha_archivo = ""                 # Texto para el nombre del archivo guardado

vmin_chla = 0.0
vmax_chla = 200.0

chla_array = np.load(ruta_npy)
chla_array_masked = np.ma.masked_where(chla_array == 0.0, chla_array) # hacemos negro los valores que sean exactamente 0.0, antes era NaN

plt.style.use('dark_background')
fig, ax = plt.subplots(1, 1, figsize=(10, 14))

im = ax.imshow(chla_array_masked, cmap='viridis', vmin=vmin_chla, vmax=vmax_chla)

ax.set_title(
    f"Clorofila-A — SENTINEL-2\n({fecha_img})\nIMAGEN PREDICHA",
    color='white', fontsize=16, fontweight='bold', pad=15
)
ax.axis('off')

# Barra de color
cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.04, shrink=0.8)
cbar.set_label('Clorofila-A (μg/L)', color='white', fontsize=12)

plt.tight_layout()


nombre_salida = f"clorofila_{fecha_archivo}_predicho.png"
plt.savefig(nombre_salida, dpi=300, bbox_inches='tight')
plt.show()